# Gold Layer — GA4 Analytics Views

## Overview

The Gold layer transforms the cleaned, flattened Silver table into **pre-aggregated analytical views** optimized for reporting and dashboards. Each view targets a specific analytical use case, grouping events by dimensions relevant to that analysis.

## Key Characteristics

| Aspect | Detail |
|---|---|
| **Source** | `workspace.silver.ga4_silver` (flattened Silver Delta table) |
| **Storage** | Views (no data duplication — computed on read) |
| **Processing** | `CREATE OR REPLACE VIEW` — idempotent, safe to rerun |
| **Use Case** | Dashboards, ad-hoc reporting, BI tools |

## Views Created

| View | Dimensions | Metric | Purpose |
|---|---|---|---|
| `GA4_daily_metrics_location` | Date, Country, Region, City | Total events | Geographic performance analysis |
| `GA4_daily_metrics_event_name` | Event name, Timestamp | Total events | Event-level traffic & conversion analysis |
| `GA4_daily_metrics_pages` | Date, Page location, Page title | Total page views | Page performance tracking |

## Processing Flow

```
Silver table (ga4_silver)
    │
    ├── GROUP BY event_date, country, region, city  →  GA4_daily_metrics_location
    ├── GROUP BY event_name, event_timestamp          →  GA4_daily_metrics_event_name
    └── GROUP BY event_date, page_location, page_title → GA4_daily_metrics_pages
```

## Views

Each section below creates a view optimized for a different analytical use case.

In [0]:
%run ./parms_config

### Geographic Performance
Country/region aggregations with engagement patterns. 

In [0]:
#Daily Traffic Summary
#Aggregated daily metrics including sessions, page views, events, engagement time, and unique pages visited
#Aggregate metrics by date including total sessions, page views, events, engagement time, and unique pages.
query1 =  f"""
        CREATE OR REPLACE VIEW {catalog}.{gold_schema}.GA4_daily_metrics_location AS
        SELECT 
        count(*) as total_events,
          event_date,
          country,
          region,
          city
        FROM {catalog}.{silver_schema}.{silver_table}
        GROUP BY event_date, country, region, city"""

spark.sql(query1)


DataFrame[]

### Daily Traffic Summary

Sequential event progression analysis for conversion optimization. 

In [0]:
query2 =  f"""
        CREATE OR REPLACE VIEW {catalog}.{gold_schema}.GA4_daily_metrics_event_name AS
        SELEct COUNT(*) AS total_events,
        event_name,
        event_timestamp
     
        FROM {catalog}.{silver_schema}.{silver_table}
        GROUP BY  event_name ,event_timestamp"""

spark.sql(query2)

DataFrame[]

### Page Performance

Page-level metrics with views, engagement time, scroll depth, and traffic sources. 

In [0]:
query3 =  f"""
        CREATE OR REPLACE VIEW {catalog}.{gold_schema}.GA4_daily_metrics_pages AS
        SELECT COUNT(*) AS total_views,
         event_date,
         page_location,
         page_title
        FROM {catalog}.{silver_schema}.{silver_table}
        GROUP BY  event_date, page_location, page_title """

spark.sql(query3)

DataFrame[]

### Device & Platform Analysis

Device category, brand, and OS metrics.

### Traffic Source Attribution

Aggregate by traffic source/medium with session quality and engagement metrics. 

In [0]:
%sql
select* from workspace.gold.ga4_daily_metrics_event_name
order by event_timestamp 
limit 100

total_events,event_name,event_timestamp
1,session_start,2026-06-06T00:38:20.994Z
1,page_view,2026-06-06T00:38:20.994Z
1,first_visit,2026-06-06T00:38:20.994Z
1,user_engagement,2026-06-06T00:38:23.776Z
1,session_start,2026-06-06T00:38:23.955Z
1,page_view,2026-06-06T00:38:23.955Z
1,first_visit,2026-06-06T00:38:23.955Z
1,scroll,2026-06-06T00:38:47.588Z
1,first_visit,2026-06-06T00:45:33.863Z
1,page_view,2026-06-06T00:45:33.863Z
